# 03 — Feature engineering

Scaffold. Use the reviewed QC series from [01](01_standardization.ipynb)
and the candidate questions from [02](02_eda.ipynb).

## Decisions to record

- Define forecast origin, target block, and horizons on the source-block index.
- Consider historical cases, lagged weather, and cumulative rainfall. Choose
  sources, lags, and windows after reviewing coverage and availability.
- Document handling of missing blocks, nulls, quality flags, and unknown
  publication timing. Specify any retrospective availability assumptions.

## Handoff

Pass a feature specification and availability checks to
[04 — Model training](04_model_training.ipynb). Check that changing rows after
a forecast origin cannot change the inputs for that origin. Fit any learned
preprocessing and selection within each training partition.

In [2]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine.upri_data import load_weekly

qc, case_column, _ = load_weekly("QUEZON_CITY")
cases = qc[case_column]
origin = cases.index

frames = []
for horizon in range(1, 5):
    target = origin + horizon
    frames.append(pd.DataFrame({
        "origin_block": origin,
        "target_block": target,
        "horizon": horizon,
        "last_observed_cases": cases.to_numpy(),
        "same_week_last_year_cases": cases.reindex(target - 52).to_numpy(),
        "actual_target_cases": cases.reindex(target).to_numpy(),
    }))

baseline_frame = pd.concat(frames, ignore_index=True).dropna().reset_index(drop=True)
rain_frame = baseline_frame.copy()

for lag in (3, 6):
    rain_block = rain_frame["target_block"] - lag
    available_at_origin = rain_block <= rain_frame["origin_block"]
    rain_frame[f"rain_lag_{lag}"] = (
        rain_block.map(qc["RF_NASA"]).where(available_at_origin)
    )

rain_coverage = rain_frame.groupby("horizon")[["rain_lag_3", "rain_lag_6"]].count()
display(rain_coverage)

,rain_lag_3,rain_lag_6
horizon,,
1,780,780
2,780,780
3,780,780
4,0,780


In [3]:
target_year = baseline_frame["target_block"] // 52
evaluation_rows = baseline_frame.loc[target_year.between(2024, 2025)].copy()
actual = evaluation_rows["actual_target_cases"]

evaluation_rows["persistence_error"] = (
    evaluation_rows["last_observed_cases"] - actual
).abs()
evaluation_rows["seasonal_error"] = (
    evaluation_rows["same_week_last_year_cases"] - actual
).abs()

baseline_scores = evaluation_rows.groupby("horizon").agg(
    forecast_weeks=("actual_target_cases", "size"),
    persistence_mae=("persistence_error", "mean"),
    seasonal_naive_mae=("seasonal_error", "mean"),
)
display(baseline_scores.round(1))

,forecast_weeks,persistence_mae,seasonal_naive_mae
horizon,,,
1,104,33.6,109.8
2,104,46.5,109.8
3,104,59.3,109.8
4,104,69.7,109.8


In [4]:
import pandas as pd

audit_rows = []

for horizon in range(1, 5):
    for lag in (3, 6):
        rain_week_relative_to_origin = horizon - lag
        audit_rows.append({
            "Forecast weeks ahead": horizon,
            "Rainfall lag": lag,
            "Rain week relative to origin": rain_week_relative_to_origin,
            "Usable with same-week report": rain_week_relative_to_origin <= 0,
            "Usable with one-week delay": rain_week_relative_to_origin <= -1,
        })

rainfall_cutoff_audit = pd.DataFrame(audit_rows)
display(rainfall_cutoff_audit)

,Forecast weeks ahead,Rainfall lag,Rain week relative to origin,Usable with same-week report,Usable with one-week delay
0,1,3,-2,True,True
1,1,6,-5,True,True
2,2,3,-1,True,True
3,2,6,-4,True,True
4,3,3,0,True,False
5,3,6,-3,True,True
6,4,3,1,False,False
7,4,6,-2,True,True
